# User Status Checker

This notebook allows the AI team to check user statuses (in/out) and their latest attendance records.

## 📦 Setup

In [ ]:
# Install dependencies (run once)
import sys
!{sys.executable} -m pip install psycopg2-binary python-dotenv pandas tabulate --quiet

print('✅ Dependencies installed!')

In [ ]:
# Imports and configuration
import psycopg2
import pandas as pd
from datetime import datetime
import os
from dotenv import load_dotenv
from tabulate import tabulate

load_dotenv()

# Database configuration
DB_CONFIG = {
    'host': 'localhost',
    'port': int(os.getenv('SO_DB_EXTERNAL_PORT', 5432)),
    'database': os.getenv('SO_DB_NAME', 'smart-office'),
    'user': os.getenv('SO_DB_USER', 'user'),
    'password': os.getenv('SO_DB_PASSWORD', 'passW0rd')
}

# Default schema (change for different organizations)
DEFAULT_SCHEMA = 'org_humblebee'

print('✅ Configuration loaded!')
print(f'   Database: {DB_CONFIG["host"]}:{DB_CONFIG["port"]}/{DB_CONFIG["database"]}')
print(f'   Default schema: {DEFAULT_SCHEMA}')

---
## 🔍 Query Functions

In [ ]:
def get_all_users_with_status(schema_name=DEFAULT_SCHEMA):
    """
    Get all users with their current status and latest attendance timestamp.

    Returns:
        pandas.DataFrame with columns:
        - id: User ID
        - full_name: User's full name
        - external_id: External employee ID
        - status: Current status (in/out)
        - last_seen: Latest attendance record timestamp
        - last_camera: Camera that last detected them
        - image_count: Number of images
    """
    conn = psycopg2.connect(**DB_CONFIG)

    query = f"""
    SELECT
        u.id,
        u.full_name,
        u.external_id,
        u.status,
        u.created_at,
        u.updated_at,
        jsonb_array_length(COALESCE(u.image_urls, '[]'::jsonb)) as image_count,
        ar.last_seen,
        ar.last_camera_id,
        c.name as last_camera_name
    FROM {schema_name}.users u
    LEFT JOIN LATERAL (
        SELECT
            timestamp as last_seen,
            camera_id as last_camera_id
        FROM {schema_name}.attendance_records
        WHERE user_id = u.id
        ORDER BY timestamp DESC
        LIMIT 1
    ) ar ON true
    LEFT JOIN {schema_name}.cameras c ON ar.last_camera_id = c.id
    ORDER BY u.id DESC;
    """

    try:
        df = pd.read_sql_query(query, conn)
        conn.close()
        return df
    except Exception as e:
        conn.close()
        raise e

print('✅ Function get_all_users_with_status() defined')

In [ ]:
def get_user_by_id(user_id, schema_name=DEFAULT_SCHEMA):
    """
    Get detailed information about a specific user.

    Args:
        user_id: User ID to lookup
        schema_name: Database schema (default: org_humblebee)

    Returns:
        dict with user details including recent attendance history
    """
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor()

    # Get user details
    cur.execute(f"""
        SELECT
            id,
            full_name,
            external_id,
            status,
            image_urls,
            created_at,
            updated_at,
            jsonb_array_length(COALESCE(image_urls, '[]'::jsonb)) as image_count
        FROM {schema_name}.users
        WHERE id = %s
    """, (user_id,))

    user = cur.fetchone()

    if not user:
        conn.close()
        return None

    # Get recent attendance records
    cur.execute(f"""
        SELECT
            ar.timestamp,
            ar.status,
            c.name as camera_name,
            c.location as camera_location
        FROM {schema_name}.attendance_records ar
        LEFT JOIN {schema_name}.cameras c ON ar.camera_id = c.id
        WHERE ar.user_id = %s
        ORDER BY ar.timestamp DESC
        LIMIT 10
    """, (user_id,))

    attendance_records = cur.fetchall()
    conn.close()

    return {
        'id': user[0],
        'full_name': user[1],
        'external_id': user[2],
        'status': user[3],
        'image_urls': user[4],
        'created_at': user[5],
        'updated_at': user[6],
        'image_count': user[7],
        'recent_attendance': attendance_records
    }

print('✅ Function get_user_by_id() defined')

In [ ]:
def get_status_summary(schema_name=DEFAULT_SCHEMA):
    """
    Get summary statistics of user statuses.

    Returns:
        dict with counts and percentages
    """
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor()

    cur.execute(f"""
        SELECT
            status,
            COUNT(*) as count
        FROM {schema_name}.users
        GROUP BY status
    """)

    results = cur.fetchall()

    # Get total count
    cur.execute(f"SELECT COUNT(*) FROM {schema_name}.users")
    total = cur.fetchone()[0]

    # Get users never tracked
    cur.execute(f"""
        SELECT COUNT(*)
        FROM {schema_name}.users u
        WHERE NOT EXISTS (
            SELECT 1 FROM {schema_name}.attendance_records ar
            WHERE ar.user_id = u.id
        )
    """)
    never_tracked = cur.fetchone()[0]

    conn.close()

    summary = {'total': total, 'never_tracked': never_tracked}
    for status, count in results:
        summary[status] = count
        summary[f'{status}_percent'] = round(count / total * 100, 1) if total > 0 else 0

    return summary

print('✅ Function get_status_summary() defined')

---
## 📊 Test 1: Get All Users with Status

In [ ]:
# Get all users
users_df = get_all_users_with_status()

print(f'📋 Total Users: {len(users_df)}')
print('\n' + '=' * 80)

# Display summary
summary = get_status_summary()
print('\n📊 STATUS SUMMARY')
print(f"   Total Users: {summary['total']}")
print(f"   Status IN:   {summary.get('in', 0)} ({summary.get('in_percent', 0)}%)")
print(f"   Status OUT:  {summary.get('out', 0)} ({summary.get('out_percent', 0)}%)")
print(f"   Never Tracked: {summary.get('never_tracked', 0)}")

# Display users table
print('\n' + '=' * 80)
print('\n👥 ALL USERS')
print('\n' + tabulate(
    users_df[['id', 'full_name', 'external_id', 'status', 'last_seen', 'last_camera_name', 'image_count']],
    headers=['ID', 'Name', 'External ID', 'Status', 'Last Seen', 'Last Camera', 'Images'],
    tablefmt='grid',
    showindex=False
))

# Also show as pandas DataFrame for easier analysis
print('\n📈 Full DataFrame:')
users_df

---
## 🔎 Test 2: Get Specific User by ID

In [ ]:
# EDIT THIS: Set the user ID you want to lookup
USER_ID = 45  # Change to any user ID

user = get_user_by_id(USER_ID)

if user:
    print(f'👤 USER DETAILS: {user["full_name"]} (ID: {user["id"]})')
    print('=' * 80)
    print(f"   External ID:     {user['external_id']}")
    print(f"   Current Status:  {user['status'].upper()}")
    print(f"   Image Count:     {user['image_count']}")
    print(f"   Created At:      {user['created_at']}")
    print(f"   Updated At:      {user['updated_at']}")

    print(f'\n📸 Image URLs:')
    if user['image_urls']:
        for i, img in enumerate(user['image_urls'], 1):
            print(f"   {i}. Original: {img.get('original', 'N/A')}")
            print(f"      Thumb:    {img.get('thumb', 'N/A')}")
    else:
        print('   No images')

    print(f'\n📅 RECENT ATTENDANCE (Last 10 records)')
    print('=' * 80)
    if user['recent_attendance']:
        attendance_data = []
        for record in user['recent_attendance']:
            attendance_data.append([
                record[0],  # timestamp
                record[1] if record[1] else 'N/A',  # status
                record[2] if record[2] else 'Unknown',  # camera_name
                record[3] if record[3] else 'N/A'  # camera_location
            ])

        print(tabulate(
            attendance_data,
            headers=['Timestamp', 'Status', 'Camera', 'Location'],
            tablefmt='grid'
        ))
    else:
        print('   No attendance records found')
else:
    print(f'❌ User with ID {USER_ID} not found')

---
## 🎯 Test 3: Filter Users by Status

In [ ]:
# Get all users
users_df = get_all_users_with_status()

# Filter by status
print('🟢 USERS CURRENTLY IN')
print('=' * 80)
users_in = users_df[users_df['status'] == 'in']
print(f'Total: {len(users_in)} users\n')
if len(users_in) > 0:
    print(tabulate(
        users_in[['id', 'full_name', 'last_seen', 'last_camera_name']],
        headers=['ID', 'Name', 'Last Seen', 'Camera'],
        tablefmt='grid',
        showindex=False
    ))
else:
    print('No users currently IN')

print('\n\n🔴 USERS CURRENTLY OUT')
print('=' * 80)
users_out = users_df[users_df['status'] == 'out']
print(f'Total: {len(users_out)} users\n')
if len(users_out) > 0:
    print(tabulate(
        users_out[['id', 'full_name', 'last_seen']].head(10),
        headers=['ID', 'Name', 'Last Seen'],
        tablefmt='grid',
        showindex=False
    ))
    if len(users_out) > 10:
        print(f'\n... and {len(users_out) - 10} more')
else:
    print('No users currently OUT')

---
## 🕐 Test 4: Users Never Tracked

In [ ]:
# Find users who have never been tracked by cameras
users_df = get_all_users_with_status()
never_tracked = users_df[users_df['last_seen'].isna()]

print('⚠️  USERS NEVER TRACKED BY CAMERAS')
print('=' * 80)
print(f'Total: {len(never_tracked)} users\n')

if len(never_tracked) > 0:
    print(tabulate(
        never_tracked[['id', 'full_name', 'external_id', 'status', 'created_at', 'image_count']],
        headers=['ID', 'Name', 'External ID', 'Status', 'Created At', 'Images'],
        tablefmt='grid',
        showindex=False
    ))
    print('\n💡 These users are enrolled but have not been detected by any camera yet.')
else:
    print('✅ All users have been tracked at least once!')

---
## 📈 Test 5: Export to CSV

In [ ]:
# Export all users to CSV for further analysis
users_df = get_all_users_with_status()

output_file = f'user_status_export_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
users_df.to_csv(output_file, index=False)

print(f'✅ Exported {len(users_df)} users to: {output_file}')
print(f'\nColumns: {", ".join(users_df.columns)}')

---
## 🔧 Advanced: Custom Queries

In [ ]:
# Example: Custom query - Users tracked in last 24 hours
conn = psycopg2.connect(**DB_CONFIG)

query = f"""
SELECT
    u.id,
    u.full_name,
    u.status,
    ar.timestamp as last_seen,
    c.name as camera_name
FROM {DEFAULT_SCHEMA}.users u
INNER JOIN {DEFAULT_SCHEMA}.attendance_records ar ON u.id = ar.user_id
LEFT JOIN {DEFAULT_SCHEMA}.cameras c ON ar.camera_id = c.id
WHERE ar.timestamp >= NOW() - INTERVAL '24 hours'
ORDER BY ar.timestamp DESC;
"""

df = pd.read_sql_query(query, conn)
conn.close()

print(f'📅 USERS TRACKED IN LAST 24 HOURS')
print('=' * 80)
print(f'Total detections: {len(df)}\n')

if len(df) > 0:
    print(tabulate(
        df.head(20),
        headers=['ID', 'Name', 'Status', 'Last Seen', 'Camera'],
        tablefmt='grid',
        showindex=False
    ))
    if len(df) > 20:
        print(f'\n... and {len(df) - 20} more detections')
else:
    print('No users tracked in the last 24 hours')

---
## 📚 Quick Reference

### Available Functions:

1. **`get_all_users_with_status(schema_name)`** - Get all users with current status and last seen timestamp
2. **`get_user_by_id(user_id, schema_name)`** - Get detailed info about specific user including recent attendance
3. **`get_status_summary(schema_name)`** - Get statistics about user statuses

### Database Schema:

**users table:**
- `id`: User ID
- `full_name`: Full name
- `external_id`: External employee ID
- `status`: Current status ('in' or 'out')
- `image_urls`: JSON array of image URLs
- `created_at`: User enrollment date
- `updated_at`: Last update timestamp

**attendance_records table:**
- `user_id`: Reference to user
- `timestamp`: When detected
- `status`: Status at detection time
- `camera_id`: Which camera detected them

### Common Use Cases:

```python
# Get all users
users = get_all_users_with_status()

# Filter by status
users_in = users[users['status'] == 'in']

# Get specific user
user = get_user_by_id(74)

# Get summary
summary = get_status_summary()
print(f"Total IN: {summary.get('in', 0)}")
```